In [1]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"

flood_path = BASE / "data/final/pp1_preprocessed/flood_human_impact_preprocessed.csv"
gis_folder = BASE / "outputs/gis"

df = pd.read_csv(flood_path)

print("=== FLOOD DATASET ===")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\n=== COLUMNS WITH MISSING VALUES ===")
missing = df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())

print("\n=== EXISTING GIS FILES ===")
for file in sorted(gis_folder.glob("*.csv")):
    print(file.name)

print("\n=== DATASET SAFETY CHECK ===")
print("Original dataset unchanged.")


=== FLOOD DATASET ===
Rows: 802
Columns: 41

COLUMN NAMES:
['District', 'Division', 'Date', 'source_record_count', 'affected_min', 'affected_max', 'affected_sum_unverified', 'source_sheets', 'target_status', 'affected_target', 'episode_number', 'episode_id', 'district_name', 'ds_name', 'census_ds_name', 'qgis_ds_name', 'ds_code', 'adm3_pcode', 'total_population', 'children_0_14_ratio', 'elderly_60_plus_ratio', 'female_ratio', 'unprotected_well_ratio', 'no_toilet_ratio', 'public_toilet_ratio', 'area_sqkm', 'road_density', 'facility_density', 'center_lat', 'center_lon', 'demographics_complete', 'gis_complete', 'pre_wl_1d', 'pre_wl_max_3d', 'pre_wl_avg_7d', 'pre_q_1d', 'pre_q_max_3d', 'pre_q_avg_7d', 'hydrology_source', 'hydrology_complete', 'risk_level']

=== COLUMNS WITH MISSING VALUES ===
pre_q_avg_7d        681
pre_q_max_3d        681
pre_q_1d            681
pre_wl_avg_7d       681
pre_wl_max_3d       681
pre_wl_1d           681
center_lat          130
center_lon          130
district

In [2]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"

flood = pd.read_csv(
    BASE / "data/final/pp1_preprocessed/flood_human_impact_preprocessed.csv"
)

gis_dir = BASE / "outputs/gis"

missing = flood[flood["center_lat"].isna()].copy()

print("=== MISSING GIS RECORDS ===")
print("Total missing:", len(missing))
print("\nBy District and Division:")
print(
    missing.groupby(["District", "Division"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)

print("\n=== EXISTING GIS MAPPING FILE STRUCTURES ===")

files = [
    "flood_ds_boundary_matching_report.csv",
    "flood_ds_district_verification.csv",
    "flood_ds_provisional_boundary_mapping.csv",
    "flood_ds_verified_priority_coordinates.csv",
]

for filename in files:
    path = gis_dir / filename
    if path.exists():
        mapping = pd.read_csv(path)
        print(f"\nFILE: {filename}")
        print("Shape:", mapping.shape)
        print("Columns:", mapping.columns.tolist())
        print("First 2 rows:")
        print(mapping.head(2).to_string(index=False))


=== MISSING GIS RECORDS ===
Total missing: 130

By District and Division:
District  Division     
Colombo   Hanwella         15
Kalutara  Walallavita      13
          Bulathsinhala    12
          Ingiriya         10
          Mathugama         9
          Palindanuwara     9
          Madurawala        8
          Panadura          8
          Bandaragama       7
          Beruwala          7
          Dodangoda         7
          Horana            7
          Agalawatta        6
          Kalutara          6
          Millaniya         6

=== EXISTING GIS MAPPING FILE STRUCTURES ===

FILE: flood_ds_boundary_matching_report.csv
Shape: (68, 7)
Columns: ['District', 'Division', 'priority_score', 'priority_level', 'ds_code', 'adm3_pcode', 'mapping_status']
First 2 rows:
 District     Division  priority_score priority_level  ds_code adm3_pcode mapping_status
  Gampaha  Attanagalla        0.749573           High     27.0     LK1227 Code available
Ratnapura Embilipitiya        0.737630   

In [3]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"
flood = pd.read_csv(BASE / "data/final/pp1_preprocessed/flood_human_impact_preprocessed.csv")
coords = pd.read_csv(BASE / "outputs/gis/flood_ds_verified_priority_coordinates.csv")

def normalize(s):
    return str(s).strip().casefold().replace(" ", "").replace("-", "")

for data in (flood, coords):
    data["_district_key"] = data["District"].map(normalize)
    data["_division_key"] = data["Division"].map(normalize)

missing = flood[flood["center_lat"].isna()].copy()

lookup = coords[
    ["_district_key", "_division_key", "center_lat", "center_lon", "gis_status", "coordinate_method"]
].drop_duplicates(["_district_key", "_division_key"])

matched = missing.merge(
    lookup,
    on=["_district_key", "_division_key"],
    how="left",
    suffixes=("_original", "_verified"),
    validate="many_to_one"
)

print("=== GIS RECOVERY CHECK ===")
print("Missing records:", len(matched))
print("With verified coordinates:", matched["center_lat_verified"].notna().sum())
print("Still without coordinates:", matched["center_lat_verified"].isna().sum())

print("\n=== BY DIVISION ===")
print(
    matched.groupby(["District", "Division"], dropna=False)
    .agg(
        records=("Date", "size"),
        recoverable=("center_lat_verified", lambda x: x.notna().sum()),
        gis_status=("gis_status", "first")
    )
    .to_string()
)

print("\nNo files modified.")


=== GIS RECOVERY CHECK ===
Missing records: 130
With verified coordinates: 100
Still without coordinates: 30

=== BY DIVISION ===
                        records  recoverable         gis_status
District Division                                              
Colombo  Hanwella            15            0                NaN
Kalutara Agalawatta           6            6  District verified
         Bandaragama          7            7  District verified
         Beruwala             7            0                NaN
         Bulathsinhala       12           12  District verified
         Dodangoda            7            7  District verified
         Horana               7            7  District verified
         Ingiriya            10           10  District verified
         Kalutara             6            6  District verified
         Madurawala           8            8  District verified
         Mathugama            9            9  District verified
         Millaniya            6       

In [4]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"
source = BASE / "data/final/pp1_preprocessed/flood_human_impact_preprocessed.csv"
mapping_path = BASE / "outputs/gis/flood_ds_verified_priority_coordinates.csv"
output = BASE / "data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv"

df = pd.read_csv(source)
mapping = pd.read_csv(mapping_path)

def normalize(value):
    return str(value).strip().casefold().replace(" ", "").replace("-", "")

for data in (df, mapping):
    data["_district_key"] = data["District"].map(normalize)
    data["_division_key"] = data["Division"].map(normalize)

gis_columns = [
    "district_name", "ds_name", "census_ds_name",
    "qgis_ds_name", "ds_code", "adm3_pcode",
    "area_sqkm", "road_density", "facility_density",
    "center_lat", "center_lon"
]

verified = mapping[
    mapping["gis_status"].eq("District verified")
].copy()

verified = verified.drop_duplicates(
    ["_district_key", "_division_key"]
)

lookup = verified[
    ["_district_key", "_division_key"] + gis_columns
]

merged = df.merge(
    lookup,
    on=["_district_key", "_division_key"],
    how="left",
    suffixes=("", "_verified"),
    validate="many_to_one"
)

before = merged["center_lat"].isna().sum()

for column in gis_columns:
    merged[column] = merged[column].fillna(
        merged[f"{column}_verified"]
    )
    merged.drop(columns=f"{column}_verified", inplace=True)

hydrology_columns = [
    "pre_wl_1d", "pre_wl_max_3d", "pre_wl_avg_7d",
    "pre_q_1d", "pre_q_max_3d", "pre_q_avg_7d"
]

merged["hydrology_missing"] = (
    merged[hydrology_columns].isna().any(axis=1)
)

merged["gis_missing"] = (
    merged[gis_columns].isna().any(axis=1)
)

merged["gis_match_status"] = "unresolved"
merged.loc[~merged["gis_missing"], "gis_match_status"] = "complete"

recovered = (
    merged["center_lat"].notna()
    & df["center_lat"].isna().to_numpy()
)
merged.loc[recovered, "gis_match_status"] = "recovered_verified"

merged["gis_complete"] = ~merged["gis_missing"]

merged.drop(
    columns=["_district_key", "_division_key"],
    inplace=True
)

assert len(merged) == len(df)
assert merged["center_lat"].isna().sum() <= before

output.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(output, index=False)

print("=== GIS RECOVERY RESULTS ===")
print("Records:", len(merged))
print("Missing GIS before:", before)
print("Missing GIS after:", merged["center_lat"].isna().sum())
print("Recovered:", recovered.sum())

print("\nGIS status:")
print(merged["gis_match_status"].value_counts().to_string())

print("\nHydrology missing:")
print(merged["hydrology_missing"].value_counts().to_string())

print("\nSaved:", output)
print("Original dataset unchanged.")


=== GIS RECOVERY RESULTS ===
Records: 802
Missing GIS before: 130
Missing GIS after: 30
Recovered: 100

GIS status:
gis_match_status
complete              672
recovered_verified    100
unresolved             30

Hydrology missing:
hydrology_missing
True     681
False    121

Saved: /Users/ohaseelanka/disaster_analytics_hub/data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv
Original dataset unchanged.


In [5]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path.home() / "disaster_analytics_hub/data/final/pp1_preprocessed"

original = pd.read_csv(base / "flood_human_impact_preprocessed.csv")
updated = pd.read_csv(base / "flood_human_impact_enriched_PP1.csv")

ml_columns = [
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio",
    "affected_target",
    "risk_level"
]

print("=== FINAL VALIDATION ===")
print("Same record count:", len(original) == len(updated))
print("ML features and targets unchanged:",
      original[ml_columns].equals(updated[ml_columns]))
print("Missing ML values:", updated[ml_columns].isna().sum().sum())
print("Duplicate full rows:", updated.duplicated().sum())

print("\n=== GIS VALIDATION ===")
print("GIS coordinates available:", updated["center_lat"].notna().sum())
print("GIS coordinates missing:", updated["center_lat"].isna().sum())
print("GIS status counts:")
print(updated["gis_match_status"].value_counts().to_string())

print("\n=== REMAINING MISSING GIS DIVISIONS ===")
print(
    updated.loc[updated["gis_missing"], ["District", "Division"]]
    .value_counts()
    .to_string()
)

print("\n=== COORDINATE RANGE CHECK ===")
has_coordinates = updated["center_lat"].notna() & updated["center_lon"].notna()
valid_coordinates = (
    updated.loc[has_coordinates, "center_lat"].between(5, 10)
    & updated.loc[has_coordinates, "center_lon"].between(79, 82)
)
print("Coordinates within Sri Lanka bounding range:",
      valid_coordinates.all())

print("\n=== HYDROLOGY VALIDATION ===")
hydro_cols = [
    "pre_wl_1d", "pre_wl_max_3d", "pre_wl_avg_7d",
    "pre_q_1d", "pre_q_max_3d", "pre_q_avg_7d"
]
print("Hydrology measurements unchanged:",
      original[hydro_cols].equals(updated[hydro_cols]))

print("\nValidation complete. No files modified.")


=== FINAL VALIDATION ===
Same record count: True
ML features and targets unchanged: True
Missing ML values: 0
Duplicate full rows: 0

=== GIS VALIDATION ===
GIS coordinates available: 772
GIS coordinates missing: 30
GIS status counts:
gis_match_status
complete              672
recovered_verified    100
unresolved             30

=== REMAINING MISSING GIS DIVISIONS ===
District  Division     
Colombo   Hanwella         15
Kalutara  Walallavita      13
          Bulathsinhala    12
          Ingiriya         10
          Mathugama         9
          Palindanuwara     9
          Panadura          8
          Madurawala        8
          Bandaragama       7
          Beruwala          7
          Dodangoda         7
          Horana            7
          Kalutara          6
          Agalawatta        6
          Millaniya         6

=== COORDINATE RANGE CHECK ===
Coordinates within Sri Lanka bounding range: True

=== HYDROLOGY VALIDATION ===
Hydrology measurements unchanged: True

Val

In [6]:

import pandas as pd
from pathlib import Path

path = (
    Path.home()
    / "disaster_analytics_hub/data/final/pp1_preprocessed"
    / "flood_human_impact_enriched_PP1.csv"
)

df = pd.read_csv(path)

gis_cols = [
    "district_name", "ds_name", "census_ds_name",
    "qgis_ds_name", "ds_code", "adm3_pcode",
    "area_sqkm", "road_density", "facility_density",
    "center_lat", "center_lon"
]

print("=== MISSING GIS ATTRIBUTES ===")
print(df[gis_cols].isna().sum().to_string())

print("\n=== GIS STATUS VS MISSINGNESS ===")
print(pd.crosstab(
    df["gis_match_status"],
    df["gis_missing"]
).to_string())

print("\n=== RECOVERED RECORDS: REMAINING MISSING FIELDS ===")
recovered = df[df["gis_match_status"] == "recovered_verified"]

print("Recovered records:", len(recovered))
print(recovered[gis_cols].isna().sum().to_string())

print("\nNo files modified.")


=== MISSING GIS ATTRIBUTES ===
district_name       130
ds_name             130
census_ds_name      130
qgis_ds_name        130
ds_code             130
adm3_pcode          130
area_sqkm           130
road_density        130
facility_density    130
center_lat           30
center_lon           30

=== GIS STATUS VS MISSINGNESS ===
gis_missing         False  True 
gis_match_status                
complete              672      0
recovered_verified      0    100
unresolved              0     30

=== RECOVERED RECORDS: REMAINING MISSING FIELDS ===
Recovered records: 100
district_name       100
ds_name             100
census_ds_name      100
qgis_ds_name        100
ds_code             100
adm3_pcode          100
area_sqkm           100
road_density        100
facility_density    100
center_lat            0
center_lon            0

No files modified.


In [7]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"
df = pd.read_csv(
    BASE / "data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv"
)

gis_attributes = [
    "district_name", "ds_name", "census_ds_name",
    "qgis_ds_name", "ds_code", "adm3_pcode",
    "area_sqkm", "road_density", "facility_density"
]

def normalize(value):
    return str(value).strip().casefold().replace(" ", "").replace("-", "")

df["_district_key"] = df["District"].map(normalize)
df["_division_key"] = df["Division"].map(normalize)

complete = df[df[gis_attributes].notna().all(axis=1)]
recovered = df[df["gis_match_status"] == "recovered_verified"]

keys = ["_district_key", "_division_key"]

lookup = complete.groupby(keys)[gis_attributes].nunique(dropna=True)

consistent = lookup[(lookup <= 1).all(axis=1)].index

available = set(consistent)
recovered_keys = list(zip(recovered["_district_key"], recovered["_division_key"]))

recoverable = sum(key in available for key in recovered_keys)

print("=== GIS ATTRIBUTE RECOVERY CHECK ===")
print("Previously recovered coordinate records:", len(recovered))
print("Records with matching complete GIS attributes:", recoverable)
print("Records without matching complete GIS attributes:", len(recovered) - recoverable)

print("\n=== BY DIVISION ===")
for (district, division), group in recovered.groupby(["District", "Division"]):
    key = (normalize(district), normalize(division))
    status = "MATCH FOUND" if key in available else "NO COMPLETE MATCH"
    print(f"{district} / {division}: {len(group)} records — {status}")

print("\nNo files modified.")


=== GIS ATTRIBUTE RECOVERY CHECK ===
Previously recovered coordinate records: 100
Records with matching complete GIS attributes: 0
Records without matching complete GIS attributes: 100

=== BY DIVISION ===
Kalutara / Agalawatta: 6 records — NO COMPLETE MATCH
Kalutara / Bandaragama: 7 records — NO COMPLETE MATCH
Kalutara / Bulathsinhala: 12 records — NO COMPLETE MATCH
Kalutara / Dodangoda: 7 records — NO COMPLETE MATCH
Kalutara / Horana: 7 records — NO COMPLETE MATCH
Kalutara / Ingiriya: 10 records — NO COMPLETE MATCH
Kalutara / Kalutara: 6 records — NO COMPLETE MATCH
Kalutara / Madurawala: 8 records — NO COMPLETE MATCH
Kalutara / Mathugama: 9 records — NO COMPLETE MATCH
Kalutara / Millaniya: 6 records — NO COMPLETE MATCH
Kalutara / Palindanuwara: 9 records — NO COMPLETE MATCH
Kalutara / Walallavita: 13 records — NO COMPLETE MATCH

No files modified.


In [8]:

from pathlib import Path

BASE = Path.home() / "disaster_analytics_hub"

keywords = [
    "gis", "spatial", "boundary", "boundaries",
    "census", "demographic", "road", "facility",
    "kalutara", "enriched", "lookup"
]

print("=== POSSIBLE GIS SOURCE FILES ===")

matches = []

for folder_name in ["data", "gis", "outputs"]:
    folder = BASE / folder_name

    if not folder.exists():
        continue

    for path in folder.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in [
            ".csv", ".xlsx", ".geojson", ".gpkg", ".shp"
        ]:
            continue

        relative_path = str(path.relative_to(BASE)).lower()

        if any(word in relative_path for word in keywords):
            matches.append(path.relative_to(BASE))

for path in sorted(set(matches)):
    print(path)

print("\nTotal matching files:", len(set(matches)))
print("No files modified.")


=== POSSIBLE GIS SOURCE FILES ===
data/backups/pp1_preprocessed/flood_human_impact_preprocessed_before_gis_fix.csv
data/final/flood_demographic_training_candidates.csv
data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv
data/processed/flood_impact_with_all_demographics.csv
data/processed/landslide_impact_with_all_demographics.csv
data/processed/landslide_impact_with_census.csv
data/raw/census/GN_housing_excel.xlsx
data/raw/census/GN_population_excel.xlsx
data/raw/census/HH_GND_excel.xlsx
gis/boundaries/sri_lanka_adm2.geojson
gis/boundaries/sri_lanka_adm3.geojson
outputs/gis/flood_ds_boundary_matching_report.csv
outputs/gis/flood_ds_boundary_review.csv
outputs/gis/flood_ds_district_verification.csv
outputs/gis/flood_ds_priority_map.geojson
outputs/gis/flood_ds_priority_map_with_seethawaka.geojson
outputs/gis/flood_ds_provisional_boundary_mapping.csv
outputs/gis/flood_ds_verified_priority_coordinates.csv

Total matching files: 18
No files modified.


In [9]:

from pathlib import Path
import pandas as pd
import json

BASE = Path.home() / "disaster_analytics_hub"

files = {
    "Flood demographic candidates":
        BASE / "data/final/flood_demographic_training_candidates.csv",
    "Processed flood demographics":
        BASE / "data/processed/flood_impact_with_all_demographics.csv",
    "GIS boundary report":
        BASE / "outputs/gis/flood_ds_boundary_matching_report.csv",
    "GIS district verification":
        BASE / "outputs/gis/flood_ds_district_verification.csv",
}

for label, path in files.items():
    print(f"\n=== {label} ===")
    data = pd.read_csv(path)
    print("Rows:", len(data))
    print("Columns:", data.columns.tolist())

    if "District" in data.columns:
        subset = data[
            data["District"].astype(str).str.strip().str.lower()
            == "kalutara"
        ]
        print("Kalutara records:", len(subset))
        print(subset.head(2).to_string(index=False))

print("\n=== ADM3 GEOJSON STRUCTURE ===")

geo_path = BASE / "gis/boundaries/sri_lanka_adm3.geojson"

with open(geo_path, encoding="utf-8") as f:
    geo = json.load(f)

print("Total features:", len(geo["features"]))
print("Properties:")
print(geo["features"][0]["properties"])

print("\nNo files modified.")



=== Flood demographic candidates ===
Rows: 802
Columns: ['District', 'Division', 'Date', 'source_record_count', 'affected_min', 'affected_max', 'affected_sum_unverified', 'source_sheets', 'target_status', 'affected_target', 'episode_number', 'episode_id', '_district_key', '_ds_key', 'district_name', 'ds_name', 'census_ds_name', 'qgis_ds_name', 'ds_code', 'adm3_pcode', 'total_population', 'children_0_14_ratio', 'elderly_60_plus_ratio', 'female_ratio', 'unprotected_well_ratio', 'no_toilet_ratio', 'public_toilet_ratio', 'area_sqkm', 'road_density', 'facility_density', 'center_lat', 'center_lon', 'demographics_complete', 'gis_complete', 'pre_wl_1d', 'pre_wl_max_3d', 'pre_wl_avg_7d', 'pre_q_1d', 'pre_q_max_3d', 'pre_q_avg_7d', 'hydrology_source', 'hydrology_complete']
Kalutara records: 115
District    Division       Date  source_record_count  affected_min  affected_max  affected_sum_unverified source_sheets target_status  affected_target  episode_number          episode_id _district_key   

In [11]:

from pathlib import Path
import pandas as pd
import geopandas as gpd

BASE = Path.home() / "disaster_analytics_hub"

verification = pd.read_csv(
    BASE / "outputs/gis/flood_ds_district_verification.csv"
)

boundaries = gpd.read_file(
    BASE / "gis/boundaries/sri_lanka_adm3.geojson"
)

kalutara = verification[
    (verification["District"] == "Kalutara")
    & (verification["verification_status"] == "District verified")
].copy()

kalutara = kalutara.merge(
    boundaries[["shapeID", "shapeName", "geometry"]],
    on="shapeID",
    how="left",
    validate="one_to_one"
)

kalutara = gpd.GeoDataFrame(
    kalutara,
    geometry="geometry",
    crs=boundaries.crs
)

# Project to a metric CRS before calculating area.
kalutara_metric = kalutara.to_crs("EPSG:32644")
kalutara["calculated_area_sqkm"] = (
    kalutara_metric.geometry.area / 1_000_000
)

print("=== VERIFIED KALUTARA BOUNDARIES ===")
print("Verified divisions:", len(kalutara))
print("Missing geometry:", kalutara.geometry.isna().sum())

print(
    kalutara[
        [
            "Division",
            "shapeName",
            "verification_status",
            "calculated_area_sqkm"
        ]
    ].to_string(index=False)
)

print("\nNo files modified.")


AttributeError: module 'fiona' has no attribute 'path'

In [12]:

from pathlib import Path
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape

BASE = Path.home() / "disaster_analytics_hub"

verification = pd.read_csv(
    BASE / "outputs/gis/flood_ds_district_verification.csv"
)

geojson_path = BASE / "gis/boundaries/sri_lanka_adm3.geojson"

# Read GeoJSON without Fiona
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson = json.load(f)

records = []

for feature in geojson["features"]:
    properties = feature["properties"].copy()
    properties["geometry"] = shape(feature["geometry"])
    records.append(properties)

boundaries = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs="EPSG:4326"
)

kalutara = verification[
    (verification["District"] == "Kalutara")
    & (verification["verification_status"] == "District verified")
].copy()

kalutara = kalutara.merge(
    boundaries[["shapeID", "shapeName", "geometry"]],
    on="shapeID",
    how="left",
    validate="one_to_one"
)

kalutara = gpd.GeoDataFrame(
    kalutara,
    geometry="geometry",
    crs="EPSG:4326"
)

kalutara_metric = kalutara.to_crs("EPSG:32644")

kalutara["calculated_area_sqkm"] = (
    kalutara_metric.geometry.area / 1_000_000
)

print("=== VERIFIED KALUTARA BOUNDARIES ===")
print("Verified divisions:", len(kalutara))
print("Missing geometry:", kalutara.geometry.isna().sum())

print(
    kalutara[
        [
            "Division",
            "shapeName",
            "verification_status",
            "calculated_area_sqkm"
        ]
    ].to_string(index=False)
)

print("\nNo files modified.")


=== VERIFIED KALUTARA BOUNDARIES ===
Verified divisions: 12
Missing geometry: 0
     Division     shapeName verification_status  calculated_area_sqkm
     Kalutara      Kalutara   District verified             75.757132
    Mathugama      Matugama   District verified            138.776455
  Bandaragama   Bandaragama   District verified             57.013546
       Horana        Horana   District verified            111.324519
Palindanuwara Palindanuwara   District verified            284.228536
    Millaniya     Millaniya   District verified             79.377763
   Madurawala    Madurawala   District verified             62.097008
   Agalawatta    Agalawatta   District verified             92.243550
  Walallavita   Walallawita   District verified            211.407142
    Dodangoda     Dodangoda   District verified            109.435607
Bulathsinhala Bulathsinhala   District verified            211.182439
     Ingiriya      Ingiriya   District verified             94.862986

No files 

In [13]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.home() / "disaster_analytics_hub"
folder = BASE / "data/final/pp1_preprocessed"

df = pd.read_csv(folder / "flood_human_impact_enriched_PP1.csv")

# Uses the verified 'kalutara' GeoDataFrame from Step 10
required = {"Division", "shapeName", "calculated_area_sqkm"}
assert "kalutara" in globals(), "Run the corrected Step 10 cell first."
assert required.issubset(kalutara.columns)

lookup = kalutara[
    ["Division", "shapeName", "calculated_area_sqkm"]
].drop_duplicates("Division")

assert lookup["Division"].is_unique

mask = (
    df["District"].eq("Kalutara")
    & df["gis_match_status"].eq("recovered_verified")
)

subset = df.loc[mask, ["Division"]].merge(
    lookup,
    on="Division",
    how="left",
    validate="many_to_one"
)

assert len(subset) == mask.sum()
assert subset["calculated_area_sqkm"].notna().all()

df.loc[mask, "district_name"] = "Kalutara"
df.loc[mask, "ds_name"] = subset["Division"].to_numpy()
df.loc[mask, "qgis_ds_name"] = subset["shapeName"].to_numpy()
df.loc[mask, "area_sqkm"] = subset["calculated_area_sqkm"].to_numpy()

# Do not invent census names, administrative codes, roads or facilities.
gis_cols = [
    "district_name", "ds_name", "census_ds_name",
    "qgis_ds_name", "ds_code", "adm3_pcode",
    "area_sqkm", "road_density", "facility_density",
    "center_lat", "center_lon"
]

df["gis_missing"] = df[gis_cols].isna().any(axis=1)
df["gis_complete"] = ~df["gis_missing"]

output = folder / "flood_human_impact_enriched_PP1_v2.csv"
df.to_csv(output, index=False)

print("=== GIS ATTRIBUTE RECOVERY ===")
print("Records updated:", mask.sum())
print("\nMissing GIS attributes:")
print(df[gis_cols].isna().sum().to_string())
print("\nFully complete GIS records:", df["gis_complete"].sum())
print("Records with coordinates:", df["center_lat"].notna().sum())
print("\nSaved:", output)
print("Previous files unchanged.")


=== GIS ATTRIBUTE RECOVERY ===
Records updated: 100

Missing GIS attributes:
district_name        30
ds_name              30
census_ds_name      130
qgis_ds_name         30
ds_code             130
adm3_pcode          130
area_sqkm            30
road_density        130
facility_density    130
center_lat           30
center_lon           30

Fully complete GIS records: 672
Records with coordinates: 772

Saved: /Users/ohaseelanka/disaster_analytics_hub/data/final/pp1_preprocessed/flood_human_impact_enriched_PP1_v2.csv
Previous files unchanged.


In [14]:

from pathlib import Path

BASE = Path.home() / "disaster_analytics_hub"

keywords = [
    "road", "highway", "hospital", "school",
    "facility", "infrastructure", "osm", "pcode"
]

extensions = {
    ".csv", ".geojson", ".gpkg", ".shp",
    ".xlsx", ".parquet", ".json", ".pbf"
}

print("=== EXISTING ROAD / FACILITY / CODE DATA SOURCES ===")

found = []

for folder_name in ["data", "gis"]:
    folder = BASE / folder_name

    if not folder.exists():
        continue

    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            if any(word in path.name.lower() for word in keywords):
                found.append(path.relative_to(BASE))

for path in sorted(set(found)):
    print(path)

print("\nTotal files found:", len(set(found)))
print("No files modified.")


=== EXISTING ROAD / FACILITY / CODE DATA SOURCES ===

Total files found: 0
No files modified.


In [15]:

from pathlib import Path
import json
import pandas as pd
from shapely.geometry import shape

BASE = Path.home() / "disaster_analytics_hub"

geo_path = BASE / "gis/boundaries/sri_lanka_adm3.geojson"
report_path = BASE / "outputs/gis/flood_ds_district_verification.csv"

with open(geo_path, encoding="utf-8") as f:
    geo = json.load(f)

records = []

for feature in geo["features"]:
    props = feature.get("properties", {})
    geometry = shape(feature["geometry"]) if feature.get("geometry") else None

    records.append({
        "shapeID": props.get("shapeID"),
        "shapeName": props.get("shapeName"),
        "geometry_valid": geometry.is_valid if geometry else False,
        "geometry_empty": geometry.is_empty if geometry else True,
        "geometry_type": geometry.geom_type if geometry else None
    })

boundaries = pd.DataFrame(records)
verification = pd.read_csv(report_path)

print("=== ORIGINAL GIS FILE AUDIT ===")
print("Total polygons:", len(boundaries))
print("Duplicate shapeIDs:", boundaries["shapeID"].duplicated().sum())
print("Missing shapeIDs:", boundaries["shapeID"].isna().sum())
print("Missing names:", boundaries["shapeName"].isna().sum())
print("Invalid geometries:", (~boundaries["geometry_valid"]).sum())
print("Empty geometries:", boundaries["geometry_empty"].sum())

print("\n=== VERIFIED KALUTARA BOUNDARIES ===")

kalutara_check = verification[
    (verification["District"] == "Kalutara")
    & (verification["verification_status"] == "District verified")
].copy()

print("Verified divisions:", len(kalutara_check))
print("Unique shapeIDs:", kalutara_check["shapeID"].nunique())
print("Duplicate assigned shapeIDs:",
      kalutara_check["shapeID"].duplicated().sum())

print("ShapeIDs missing from GeoJSON:",
      (~kalutara_check["shapeID"].isin(boundaries["shapeID"])).sum())

print("\nNo files modified.")


=== ORIGINAL GIS FILE AUDIT ===
Total polygons: 330
Duplicate shapeIDs: 0
Missing shapeIDs: 0
Missing names: 0
Invalid geometries: 0
Empty geometries: 0

=== VERIFIED KALUTARA BOUNDARIES ===
Verified divisions: 12
Unique shapeIDs: 12
Duplicate assigned shapeIDs: 0
ShapeIDs missing from GeoJSON: 0

No files modified.


In [16]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"
folder = BASE / "data/final/pp1_preprocessed"

df = pd.read_csv(folder / "flood_human_impact_enriched_PP1_v2.csv")

columns = [
    "District", "Division", "Date", "episode_id",
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio",
    "affected_target",
    "risk_level"
]

assert set(columns).issubset(df.columns)

presentation = df[columns].copy()

assert presentation.isna().sum().sum() == 0
assert not presentation.duplicated().any()
assert presentation["risk_level"].nunique() == 4
assert presentation["affected_target"].ge(0).all()

output = folder / "flood_ML_dataset_PP1_FINAL.csv"
presentation.to_csv(output, index=False)

print("=== PP1 FLOOD DATASET ===")
print("Rows:", len(presentation))
print("Columns:", len(presentation.columns))
print("Missing values:", presentation.isna().sum().sum())
print("Duplicate rows:", presentation.duplicated().sum())

print("\nRisk categories:")
print(presentation["risk_level"].value_counts().to_string())

print("\nPreview:")
print(presentation.head(5).to_string(index=False))

print("\nSaved:", output)


=== PP1 FLOOD DATASET ===
Rows: 802
Columns: 13
Missing values: 0
Duplicate rows: 0

Risk categories:
risk_level
Moderate     207
Low          199
Very High    199
High         197

Preview:
 District  Division       Date           episode_id  total_population  children_0_14_ratio  elderly_60_plus_ratio  female_ratio  unprotected_well_ratio  no_toilet_ratio  public_toilet_ratio  affected_target risk_level
Ratnapura Balangoda 2001-04-21 FLOOD_Ratnapura_0003           74893.0             0.214760               0.183956      0.513933                0.016309         0.001128             0.000799             80.0   Moderate
Ratnapura   Kolonna 2001-07-20 FLOOD_Ratnapura_0004           50068.0             0.221039               0.183890      0.501937                0.023765         0.000606             0.002087            135.0   Moderate
Ratnapura Opanayaka 2003-02-15 FLOOD_Ratnapura_0009           27244.0             0.202430               0.192189      0.514939                0.031331    